# Optogenetic Figures

Interactive notebook for CsChrimson Figure 4 and GtACR Figure 5. Run the setup cells first, then enable only the figure cells you need.

## 1. Environment and plotting setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from contextlib import contextmanager
from pathlib import Path
from IPython.display import IFrame, display
import os
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "KinematicPlot.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Repository root: {ROOT}")
print(f"Python executable: {sys.executable}")

In [ ]:
import KinematicPlot as kp
from group_config_new import build_groups
from survival_stats_runner import SurvivalStatsRunner

FIGURES_DIR = ROOT / "Figures"
NOTEBOOK_OUTPUT_DIR = FIGURES_DIR / "OptogeneticFigure"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_PERM = 20000  # Use 1000 while testing and 20000 for final statistics.

plotter = kp.PlotCreator(
    platform_offset=0.07,
    platform_height=0.1,
    radius=0.03,
    fps=250,
)
stats_runner = SurvivalStatsRunner(
    tau=0.71,
    random_state=0,
    platform_offset=0.03,
    radius=0.07,
    fps=250,
)

def output_folder(*parts):
    folder = NOTEBOOK_OUTPUT_DIR.joinpath(*parts)
    folder.mkdir(parents=True, exist_ok=True)
    return folder

@contextmanager
def working_directory(folder):
    previous = Path.cwd()
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    os.chdir(folder)
    try:
        yield folder
    finally:
        os.chdir(previous)

def show_pdf(path, width=950, height=700):
    path = Path(path)
    try:
        source = path.relative_to(ROOT).as_posix()
    except ValueError:
        source = path.as_posix()
    display(IFrame(src=source, width=width, height=height))

def show_all_pdfs(folder, height=650):
    for pdf in sorted(Path(folder).glob("*.pdf")):
        print(pdf.name)
        show_pdf(pdf, height=height)

print(f"Notebook outputs: {NOTEBOOK_OUTPUT_DIR}")

## 2. Configure and build optogenetic groups

In [ ]:
chr_low_keys = [
    "ADxChr-400uW",
    "IavxChr-400uW",
    "HP2xChr-400uW",
    "TaCSxCHR-400uW",
    "AllCSxChr-400uW",
]
chr_med_keys = [
    "IAVxCHR-4mW",
    "HP2xCHR-4mW",
    "TaBriLexAR-4mW",
    "TaCSxCHR-4mW",
    "CSS0048xCHR-4mW",
    "BiCSxCHR-4mW",
    "BiCS-HaltxCHR-4mW",
]
chr_high_keys = [
    "IAVxCHR-12mW",
    "HP2xChr-12mW",
    "TaBriLexAR-12mW",
    "TaCSxCHR-12mW",
    "CSS0048xCHR-12mW",
    "BiCS-HaltWgxCHR-12mW",
    "BICSxCHR-12mW",
    "BICSHALTxCHR-12mW",
    "CSS0021xCHR-12mW",
]
gtacr_keys = [
    "WT_Green",
    "LexA_Br",
    "MTGal4",
    "IavxGTACR",
    "CSS048xGTACR",
    "CSS021xGTACR",
]

all_opto_keys = list(dict.fromkeys(chr_low_keys + chr_med_keys + chr_high_keys + gtacr_keys))
groups = build_groups(
    group_keys=all_opto_keys,
    skip_missing=False,
    require_kinematics=False,
)

pd.DataFrame([
    {
        "Config_Key": key,
        "Group_Name": group.group_name,
        "Metadata_Loaded": len(group.trial_metadata),
        "Kinematic_Trials_Loaded": len(group.fly_kinematic_data),
    }
    for key, group in groups.items()
])

In [ ]:
chr_intensity_blocks = {
    "low": {"lp_label": "4A", "ll_label": "4D", "keys": chr_low_keys},
    "medium": {"lp_label": "4B", "ll_label": "4E", "keys": chr_med_keys},
    "high": {"lp_label": "4C", "ll_label": "4F", "keys": chr_high_keys},
}

chr_lp_change_keys = [
    "ADxChr-400uW",
    "IavxChr-400uW", "IAVxCHR-4mW", "IAVxCHR-12mW",
    "HP2xChr-400uW", "HP2xCHR-4mW", "HP2xChr-12mW",
    "AllCSxChr-400uW",
    "BiCSxCHR-4mW", "BICSxCHR-12mW",
    "BiCS-HaltxCHR-4mW", "BICSHALTxCHR-12mW",
    "BiCS-HaltWgxCHR-12mW",
    "CSS0048xCHR-4mW", "CSS0048xCHR-12mW",
    "CSS0021xCHR-12mW",
    "TaCSxCHR-400uW", "TaCSxCHR-4mW", "TaCSxCHR-12mW",
    "TaBriLexAR-4mW", "TaBriLexAR-12mW",
]

chrimson_selected_keys = [
    "ADxChr-400uW",
    "AllCSxChr-400uW",
    "CSS0048xCHR-12mW",
    "TaBriLexAR-4mW",
    "CSS0021xCHR-12mW",
]
chrimson_selected_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "#8c564b"]

gtacr_colors = {
    "WT_Green": "black",
    "LexA_Br": "green",
    "MTGal4": "blue",
    "IavxGTACR": "brown",
    "CSS048xGTACR": "red",
    "CSS021xGTACR": "orange",
}

# Figure 4: CsChrimson

## Figures 4A and 4D: Low-intensity LP and ON latency

In [ ]:
RUN_CHR_LOW = False

if RUN_CHR_LOW:
    lp_out = output_folder("CsChrimson_Figure_4", "4A_low_landing_probability")
    ll_data = []
    with working_directory(lp_out):
        for group_key in chr_low_keys:
            ll_data.append(plotter.plot_chrimson_LP(groups[group_key], color="red", threshold=0.71))

    km_out = output_folder("CsChrimson_Figure_4", "4D_low_landing_latency_KM")
    km_pdf = km_out / "Figure_4D_CsChrimson_low_ON_landing_latency_KM.pdf"
    plotter.plot_kmc_and_unpaired_rmst_perm(
        data_list=ll_data,
        file_name=str(km_pdf.with_suffix("")),
        tau=0.71,
        n_perm=N_PERM,
        random_state=0,
    )
    show_all_pdfs(lp_out)
    show_pdf(km_pdf)

## Figures 4B and 4E: Medium-intensity LP and ON latency

In [ ]:
RUN_CHR_MEDIUM = False

if RUN_CHR_MEDIUM:
    lp_out = output_folder("CsChrimson_Figure_4", "4B_medium_landing_probability")
    ll_data = []
    with working_directory(lp_out):
        for group_key in chr_med_keys:
            ll_data.append(plotter.plot_chrimson_LP(groups[group_key], color="red", threshold=0.71))

    km_out = output_folder("CsChrimson_Figure_4", "4E_medium_landing_latency_KM")
    km_pdf = km_out / "Figure_4E_CsChrimson_medium_ON_landing_latency_KM.pdf"
    plotter.plot_kmc_and_unpaired_rmst_perm(
        data_list=ll_data,
        file_name=str(km_pdf.with_suffix("")),
        tau=0.71,
        n_perm=N_PERM,
        random_state=0,
    )
    show_all_pdfs(lp_out)
    show_pdf(km_pdf)

## Figures 4C and 4F: High-intensity LP and ON latency

In [ ]:
RUN_CHR_HIGH = False

if RUN_CHR_HIGH:
    lp_out = output_folder("CsChrimson_Figure_4", "4C_high_landing_probability")
    ll_data = []
    with working_directory(lp_out):
        for group_key in chr_high_keys:
            ll_data.append(plotter.plot_chrimson_LP(groups[group_key], color="red", threshold=0.71))

    km_out = output_folder("CsChrimson_Figure_4", "4F_high_landing_latency_KM")
    km_pdf = km_out / "Figure_4F_CsChrimson_high_ON_landing_latency_KM.pdf"
    plotter.plot_kmc_and_unpaired_rmst_perm(
        data_list=ll_data,
        file_name=str(km_pdf.with_suffix("")),
        tau=0.71,
        n_perm=N_PERM,
        random_state=0,
    )
    show_all_pdfs(lp_out)
    show_pdf(km_pdf)

## Figures 4A-4C: Combined fly-wise LP change summary

In [ ]:
RUN_CHR_LP_CHANGE_SUMMARY = False

if RUN_CHR_LP_CHANGE_SUMMARY:
    out = output_folder("CsChrimson_Figure_4", "4A_4C_combined_LP_change_summary")
    pdf = out / "Figure_4A_4C_CsChrimson_combined_LP_change_ON_minus_OFF.pdf"
    plotter.plot_chrimson_LP_change_summary(
        groups={key: groups[key] for key in chr_lp_change_keys},
        file_name=str(pdf.with_suffix("")),
        threshold=0.71,
        n_perm=N_PERM,
        random_state=0,
    )
    show_pdf(pdf, height=950)

## Selected groups: Individual ON/OFF inverted latency curves

In [ ]:
RUN_CHR_SELECTED_ON_OFF_KM = False

if RUN_CHR_SELECTED_ON_OFF_KM:
    out = output_folder("CsChrimson_Figure_4", "4_selected_groups_ON_OFF_landing_latency_KM")
    for group_key, color in zip(chrimson_selected_keys, chrimson_selected_colors):
        pdf = out / f"Figure_4_{group_key}_ON_OFF_landing_latency_KM.pdf"
        plotter.plot_chrimson_on_off_latency_km(
            group_info=groups[group_key],
            file_name=str(pdf.with_suffix("")),
            color=color,
            tau=0.71,
        )
    show_all_pdfs(out)

## Selected groups: ON/OFF R-mFT and wing angle traces

In [ ]:
RUN_CHR_SELECTED_ANGLE_TRACES = False

if RUN_CHR_SELECTED_ANGLE_TRACES:
    out = output_folder("CsChrimson_Figure_4", "4_selected_groups_ON_OFF_angle_traces")
    for group_key, color in zip(chrimson_selected_keys, chrimson_selected_colors):
        pdf = out / f"Figure_4_{group_key}_ON_OFF_RmFT_wing_angle_traces.pdf"
        plotter.plot_chrimson_on_off_leg_wing_angle_traces(
            group_info=groups[group_key],
            angles=[
                ["R-mCT", "R-mFT", "R-mTT"],
                ["wing", "wing", "wing"],
            ],
            file_name=str(pdf.with_suffix("")),
            start=-0.5,
            end=3,
            color=color,
            show_sem=True,
        )
    show_all_pdfs(out)

## Figures 4G-4I: Per-group leg and wing angle traces by intensity

In [ ]:
RUN_CHR_INTENSITY_ANGLE_TRACES = False

if RUN_CHR_INTENSITY_ANGLE_TRACES:
    angle_blocks = {
        "4G_low_angle_traces": chr_low_keys,
        "4H_medium_angle_traces": chr_med_keys,
        "4I_high_angle_traces": chr_high_keys,
    }
    for block_name, group_keys in angle_blocks.items():
        out = output_folder("CsChrimson_Figure_4", block_name)
        with working_directory(out):
            for group_key in group_keys:
                plotter.plot_group_angle_trace_opto(
                    group_info=groups[group_key],
                    angles=[
                        ["R-mCT", "R-mFT", "R-mTT"],
                        ["wing", "wing", "wing"],
                    ],
                    start=-0.5,
                    end=3,
                    colors="red",
                    chrimson=True,
                )
        show_all_pdfs(out)

# Figure 5: GtACR

## Figure 5A: Individual ON/OFF landing probability

In [ ]:
RUN_GTACR_INDIVIDUAL_LP = False

if RUN_GTACR_INDIVIDUAL_LP:
    out = output_folder("GtACR_Figure_5", "5A_landing_probability_ON_OFF")
    with working_directory(out):
        for group_key in gtacr_keys:
            plotter.plot_LP_summary_light_from_group(
                group_info=groups[group_key],
                file_name=f"Figure_5A_{group_key}_LP_ON_OFF",
                color="green",
            )
    show_all_pdfs(out)

## Figure 5A: Combined fly-wise LP change summary

In [ ]:
RUN_GTACR_LP_CHANGE_SUMMARY = False

if RUN_GTACR_LP_CHANGE_SUMMARY:
    out = output_folder("GtACR_Figure_5", "5A_combined_LP_change_summary")
    pdf = out / "Figure_5A_GtACR_combined_LP_change_ON_minus_OFF.pdf"
    plotter.plot_gtacr_LP_change_summary(
        groups={key: groups[key] for key in gtacr_keys},
        file_name=str(pdf.with_suffix("")),
        n_perm=N_PERM,
        random_state=0,
    )
    show_pdf(pdf)

## Figure 5B: Individual ON/OFF landing-latency KM curves

In [ ]:
RUN_GTACR_KM_CURVES = False

if RUN_GTACR_KM_CURVES:
    out = output_folder("GtACR_Figure_5", "5B_landing_latency_KM_ON_OFF")
    for group_key in gtacr_keys:
        pdf = out / f"Figure_5B_{group_key}_LL_KM_ON_OFF.pdf"
        plotter.plot_KM_curve_from_groups(
            groups=[groups[group_key]],
            file_name=str(pdf.with_suffix("")),
            colors=["black", "green"],
            opto=True,
        )
    show_all_pdfs(out)

## Figure 5: Paired ON/OFF LP and latency statistics

In [ ]:
RUN_GTACR_STATS = False

if RUN_GTACR_STATS:
    out = output_folder("GtACR_Figure_5", "stats")
    lp_rows = []
    ll_rows = []
    for group_key in gtacr_keys:
        group_info = groups[group_key]
        lp_result, _ = stats_runner.compare_lp_paired(
            group_info=group_info,
            out_prefix=str(out / f"Figure_5_GtACR_{group_key}_LP_ON_OFF"),
            on_label="ON",
            off_label="OFF",
            n_perm=N_PERM,
        )
        lp_rows.append(lp_result)

        ll_result, _ = stats_runner.analyze_landing_opto(
            group_info=group_info,
            out_prefix=str(out / f"Figure_5_GtACR_{group_key}_LL_ON_OFF"),
            chr_data=False,
            n_perm=N_PERM,
        )
        ll_rows.append(ll_result)

    lp_df = pd.concat(lp_rows, ignore_index=True)
    ll_df = pd.concat(ll_rows, ignore_index=True)
    lp_df.to_csv(out / "Figure_5_GtACR_LP_ON_OFF_stats.csv", index=False)
    ll_df.to_csv(out / "Figure_5_GtACR_LL_ON_OFF_fly_RMST_stats.csv", index=False)
    display(lp_df)
    display(ll_df)

## Usage notes

1. Run Sections 1 and 2 after starting or restarting the kernel.
2. Set only the desired cell's `RUN_...` flag to `True`.
3. Return the flag to `False` afterward to avoid accidental reruns.
4. Outputs are saved under `Figures/OptogeneticFigure` and PDFs are embedded below their cells.
5. Some legacy plotting functions save relative to the current working directory; the notebook handles this with `working_directory(...)`.
6. Use `N_PERM = 1000` during debugging and restore `20000` for final statistics.